In [ ]:
import os, sys
from pathlib import Path

print('=== cwd ===')
print(Path.cwd())

print()
print('=== home dir contents ===')
for p in sorted(Path.home().iterdir()):
    print(' ', p)

print()
print('=== sys.path ===')
for p in sys.path:
    print(' ', p)

print()
print('=== SYNTHGEN_ROOT env var ===')
print(os.environ.get("SYNTHGEN_ROOT", "(not set)"))

In [1]:
# Run on GPU server (Jupyter). Local CPU will OOM at L=2520.
import os
import sys
from pathlib import Path

import torch

# Locate repo root: env var > ~/SyntheticGenerators > cwd > cwd parent
_env = os.environ.get("SYNTHGEN_ROOT", "")
REPO = Path(_env) if _env else None
if REPO is None or not (REPO / "SBBTS").is_dir():
    for _c in [
        Path.home() / "SyntheticGenerators",   # GPUHub default clone location
        Path.cwd(),
        Path.cwd().parent,
    ]:
        if (_c / "SBBTS").is_dir() and (_c / "data").is_dir():
            REPO = _c
            break
    else:
        raise RuntimeError(
            "Repo not found. Set SYNTHGEN_ROOT env var or clone to ~/SyntheticGenerators."
        )

SBBTS = REPO / "SBBTS"
for p in (SBBTS, REPO):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required for SBBTS training (seq_len=2520).")

device = torch.device("cuda")
OUTPUT_DIR = REPO / "data" / "output_data"
TRAIN_NPY = OUTPUT_DIR / "train_normalized.npy"
CHECKPOINT_DIR = SBBTS / "checkpoints" / "sbbts_run"

print(f"repo        = {REPO}")
print(f"SBBTS       = {SBBTS}")
print(f"TRAIN_NPY   = {TRAIN_NPY}  (exists={TRAIN_NPY.exists()})")
print(f"device      = {device} ({torch.cuda.get_device_name(0)})")

repo        = /home/jovyan/SyntheticGenerators
SBBTS       = /home/jovyan/SyntheticGenerators/SBBTS
TRAIN_NPY   = /home/jovyan/SyntheticGenerators/data/output_data/train_normalized.npy  (exists=False)
device      = cuda (NVIDIA A16)


In [2]:
from adapter_sbbts import build_sbbts_tensor, sample_synthetic, train_sbbts

# 14-16GB GPU: use 1-2. 24GB+: try 4. Restart kernel after pulling adapter changes.
BATCH_SIZE = 2
N_EPOCHS = 1000

X, scale, idx, meta = build_sbbts_tensor(
    train_npy_path=TRAIN_NPY,
    M_train=500,
    device=device,
)

model, y_0 = train_sbbts(
    X=X,
    scale=scale,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
)

synthetic = sample_synthetic(
    X=X,
    model=model,
    y_0=y_0,
    scale=scale,
    meta=meta,
    output_dir=OUTPUT_DIR,
    M_simu=200,
    device=device,
)
print("benchmark export:", OUTPUT_DIR / "sbbts_synthetic.npy")


AssertionError: Not found: /home/jovyan/SyntheticGenerators/data/output_data/train_normalized.npy

In [ ]:
# Optional: smoke test (2 epochs) before the full run in cell 1
from adapter_sbbts import train_sbbts

model, y_0 = train_sbbts(
    X=X,
    scale=scale,
    checkpoint_dir=CHECKPOINT_DIR / "smoke",
    device=device,
    batch_size=2,
    n_epochs=2,
)